In [4]:
!pip install implicit

In [ ]:
import os
import gc
import re
import json
import argparse
import pickle
import time
import psutil
import numpy as np
import polars as pl
import scipy.sparse as sp
from datetime import datetime
from pathlib import Path
from scipy.sparse import csr_matrix, coo_matrix
from scipy.sparse.linalg import svds
import implicit  # Thư viện cho luồng Collaborative Filtering ALS chuyên sâu
import pyarrow as pa
import pyarrow.parquet as pq
import lightgbm as lgb

# ==============================================================================
# CONFIGURATION & BASE PATHS
# ==============================================================================
ROOT = Path("/kaggle/input/datasets/xunthanhh/cs-116")
TRANSACTION_PATH = ROOT / "transaction_full_2025.parquet"
EVENT_PATH = ROOT / "event_full_2025.parquet"
ITEMS_PATH = ROOT / "items.parquet"

CACHE_DIR = Path("./recs_cache")
CACHE_DIR.mkdir(exist_ok=True)

# HẰNG SỐ ĐIỀU TỐC KIỂM THỬ TOÀN CỤC (-1 ĐỂ CHẠY FULL SCALE TOÀN BỘ ~3 TRIỆU USER)
USER_LIMIT = 10000  

# CÔNG TẮC NGẶT MẠCH CHỐNG OOM TUYỆT ĐỐI
ENABLE_COBUY = False  

# MẢNG CÁC ĐẶC TRƯNG TỔ HỢP BAO GỒM KÊNH THÔ VÀ ĐẶC TRƯNG HÀNH VI CHAMPION ĐỂ MODEL HỌC TẬP
CHANNELS = ["A_history", "C_svd", "D_i2i", "S2_als", "B_local", "E_cat", "F_brand", "H_trend", "G_global", "S4_month_pop", "S5_month_trend", "S1_hist_recent", "S6_full_hist"]
BEHAVIOR_FEATURES = [
    "u_total_tx", "u_total_qty", "u_total_spend", "u_unique_items", "u_tenure_days", "u_recency_days",
    "u_loc_hhi", "u_cat_hhi", "u_brand_hhi", "u_avg_basket_size", "u_avg_order_value", "u_tx_velocity",
    "price", "i_sales_all", "i_sales_30d", "i_sales_60_to_30d", "i_sales_90_to_60d",
    "cat_sales_30d", "cat_sales_60d", "cat_momentum", "i_momentum_30d", "i_momentum_60d",
    "item_repeat_propensity", "item_launch_age_days",
    "ui_purchase_count", "ui_total_qty", "ui_days_since_last", "ui_days_since_first",
    "ui_avg_qty_per_order", "ui_purchase_velocity", "ui_replenishment_due", "ui_age_delta", "ui_recency_weight",
    "cross_cat_momentum", "cross_brand_momentum", "cross_price_ratio", "ui_habitual_match",
    "basket_size_mismatch", "weekend_shopper_match", "replenishment_overdue_days", 
    "svd_similarity", "price_ratio", "u_avg_price", "item_avg_price", "i_total_sales", "svd_score"
]
LGBM_FEATURES = CHANNELS + BEHAVIOR_FEATURES

def print_status(msg: str):
    """Hàm ghi nhận mốc thời gian và lượng RAM tiêu thụ thực tế tại thời điểm gọi"""
    process = psutil.Process(os.getpid())
    ram_gb = process.memory_info().rss / (1024 ** 3)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] [RAM: {ram_gb:.2f} GB] {msg}")

def standardize_age(text):
    """Hàm trích xuất phân tích văn bản mô tả để quy đổi độ tuổi nhi đồng bằng chuỗi luật Regex"""
    raw_text = str(text).strip()
    clean_text = raw_text.lower()
    if re.search(r'(\*|x\d|cm)', clean_text): return None
    if re.search(r'\bb\d{2}\b', clean_text): return None
    if 's17' in clean_text: return 1.0
    if '110' in clean_text: return 5.0
    if "không xác định" in clean_text or not clean_text or clean_text == "none": return None
    diaper_map = {
        r'\bnb\b': 0.0, r'\bss\b': 0.0, r'\bsơ sinh\b': 0.0,
        r'\bs\b': 0.25, r'\bm\b': 0.6, r'\bl\b': 1.2,
        r'\bxl\b': 2.0, r'\bxxl\b': 3.5
    }
    for pattern, val in diaper_map.items():
        if re.search(pattern, clean_text): return val
    range_match = re.search(r'(\d+\.?\d*)\s*-\s*(\d+\.?\d*)', clean_text)
    if range_match:
        s, e = float(range_match.group(1)), float(range_match.group(2))
        avg = (s + e) / 2
        if any(x in clean_text for x in ['m', 'tháng']): return round(avg / 12, 3)
        return avg
    m_match = re.search(r'(\d+\.?\d*)\s*(m|tháng)', clean_text)
    if m_match: return round(float(m_match.group(1)) / 12, 3)
    y_match = re.search(r'(\d+\.?\d*)\s*(y|t|tuổi)', clean_text)
    if y_match: return float(y_match.group(1))
    pure_num = re.search(r'^(\d+)$', clean_text)
    if pure_num:
        val = float(pure_num.group(1))
        return round(val/12, 3) if val > 6 else val
    return None

def load_items() -> pl.DataFrame:
    print_status("Loading items metadata and executing description age parsing text mining...")
    df = pl.read_parquet(ITEMS_PATH)
    cols = ["item_id", "category_l1", "category_l2", "category_l3", "brand", "price", "description", "sale_status"]
    df = df.select([c for c in cols if c in df.columns]).with_columns([
        pl.col("item_id").cast(pl.Utf8),
        pl.col("category_l1").cast(pl.Utf8).fill_null("Unknown"),
        pl.col("category_l2").cast(pl.Utf8).fill_null("Unknown"),
        pl.col("category_l3").cast(pl.Utf8).fill_null("Unknown"),
        pl.col("brand").cast(pl.Utf8).fill_null("Unknown"),
        pl.col("price").cast(pl.Float32).fill_null(0.0),
        pl.col("sale_status").cast(pl.Int32).fill_null(0),
    ])
    if "description" in df.columns:
        df = df.with_columns(
            pl.col("description").cast(pl.Utf8).fill_null("Unknown")
                .map_elements(standardize_age, return_dtype=pl.Float64)
                .cast(pl.Float32).alias("i_target_age_years")
        ).drop("description")
    else:
        df = df.with_columns(pl.lit(None).cast(pl.Float32).alias("i_target_age_years"))
    return df

def load_history_data() -> tuple[pl.DataFrame, pl.DataFrame]:
    print_status("Loading full transaction history (All Available Months) and extracting target users...")
    hist_tx = (
        pl.read_parquet(TRANSACTION_PATH)
        .select([
            pl.col("customer_id").cast(pl.Int32),
            pl.col("item_id").cast(pl.Utf8),
            pl.col("quantity").cast(pl.Float32).fill_null(1.0),
            pl.col("location").cast(pl.Int32),
            pl.col("updated_date").cast(pl.Datetime).alias("event_ts"),
            pl.col("bill_id"),
        ])
    )
    target_users = hist_tx.select("customer_id").unique()
    print_status(f"Extracted {target_users.height:,} unique target users from history.")
    return hist_tx, target_users

def load_event_history(target_users: pl.DataFrame) -> pl.DataFrame:
    print_status("Loading full event interaction history (Views, Carts)...")
    return (
        pl.scan_parquet(EVENT_PATH)
        .filter(
            pl.col("event_type").is_in(["view_item", "add_to_cart"]) & 
            pl.col("item_id").is_not_null()
        )
        .select([
            pl.col("customer_id").cast(pl.Int32),
            pl.col("item_id").cast(pl.Utf8),
            pl.col("event_date").cast(pl.Datetime).alias("event_ts")
        ])
        .join(target_users.lazy(), on="customer_id", how="inner")
        .collect()
    )

# ==============================================================================
# ADVANCED FEATURE EXTRACTION (100% POLARS NATIVE - ZERO PANDAS)
# ==============================================================================
def precompute_advanced_features(hist_tx: pl.DataFrame, items: pl.DataFrame) -> tuple[Path, Path, int]:
    print_status("Precomputing advanced filtering features (User-Item Price Metrics & Category SVD)...")
    
    hist_joined = hist_tx.join(items.select(["item_id", "category_l2"]), on="item_id", how="left")
    user_cat_matrix = (
        hist_joined.group_by(["customer_id", "category_l2"])
        .agg(pl.len().alias("purchases"))
    )
    
    unique_users_svd = user_cat_matrix.select("customer_id").unique()["customer_id"].to_numpy()
    unique_cats_svd = user_cat_matrix.select("category_l2").drop_nulls().unique()["category_l2"].to_numpy()
    
    uc_valid = user_cat_matrix.filter(pl.col("category_l2").is_not_null())
    
    k_svd = 10
    if len(unique_users_svd) > k_svd and len(unique_cats_svd) > k_svd:
        user_map_df = pl.DataFrame({"customer_id": unique_users_svd, "user_idx": np.arange(len(unique_users_svd), dtype=np.int32)})
        cat_map_df = pl.DataFrame({"category_l2": unique_cats_svd, "cat_idx": np.arange(len(unique_cats_svd), dtype=np.int32)})
        
        uc_indexed = uc_valid.join(user_map_df, on="customer_id").join(cat_map_df, on="category_l2")
        row_svd = uc_indexed["user_idx"].to_numpy()
        col_svd = uc_indexed["cat_idx"].to_numpy()
        data_svd = uc_indexed["purchases"].to_numpy()
        
        mat_svd = coo_matrix((data_svd, (row_svd, col_svd)), shape=(len(unique_users_svd), len(unique_cats_svd))).astype(np.float32)
        
        print_status("Executing svds decomposition...")
        U, S, Vt = svds(mat_svd, k=k_svd)
        sort_idx = np.argsort(S)[::-1]
        U = U[:, sort_idx]
        Vt = Vt[sort_idx, :]
        
        U_df = pl.DataFrame(U, schema=[f"u_svd_{i}" for i in range(k_svd)]).with_columns(pl.Series("customer_id", unique_users_svd, dtype=pl.Int32))
        V_df = pl.DataFrame(Vt.T, schema=[f"c_svd_{i}" for i in range(k_svd)]).with_columns(pl.Series("category_l2", unique_cats_svd))
    else:
        k_svd = 0
        U_df = pl.DataFrame({"customer_id": pl.Series(unique_users_svd, dtype=pl.Int32)})
        V_df = pl.DataFrame({"category_l2": pl.Series(unique_cats_svd, dtype=pl.Utf8)})

    hist_with_price = hist_tx.join(items.select(["item_id", "price"]), on="item_id", how="left")

    print_status("Building User Feature Base...")
    user_features = (
        hist_with_price.group_by("customer_id").agg([pl.col("price").mean().alias("u_avg_price")])
        .join(U_df, on="customer_id", how="left")
    )
    user_features = user_features.with_columns([
        pl.col(c).fill_null(0.0) for c in user_features.columns if c != "customer_id"
    ])
    
    print_status("Building Item Feature Base...")
    item_features = (
        hist_with_price.group_by("item_id").agg([
            pl.col("price").mean().alias("item_avg_price"),
            pl.len().alias("i_total_sales")
        ])
        .join(items.select(["item_id", "category_l2"]), on="item_id", how="left")
        .join(V_df, on="category_l2", how="left").drop("category_l2")
    )
    item_features = item_features.with_columns([
        pl.col(c).fill_null(0.0) for c in item_features.columns if c != "item_id"
    ])
    
    user_features_path = CACHE_DIR / "user_features.parquet"
    item_features_path = CACHE_DIR / "item_features.parquet"
    user_features.write_parquet(user_features_path)
    item_features.write_parquet(item_features_path)
    
    del hist_joined, user_cat_matrix, uc_valid, hist_with_price, U_df, V_df, user_features, item_features, user_map_df, cat_map_df, uc_indexed; gc.collect()
    return user_features_path, item_features_path, k_svd

# ==============================================================================
# PROFILE EXTRACTION CHANNELS & RICH LOOKUP TABLES
# ==============================================================================
def compute_user_archetypes_and_profiles(hist_tx: pl.DataFrame, items: pl.DataFrame, target_users: pl.DataFrame) -> dict:
    print_status("Computing deep behavioral lookup archetypes and rich features (HHI, Basket Sizes, Temporal)...")
    max_ts = hist_tx["event_ts"].max()
    t_30 = max_ts - pl.duration(days=30)
    t_60 = max_ts - pl.duration(days=60)
    t_90 = max_ts - pl.duration(days=90)
    
    item_sub = items.select(["item_id", "category_l1", "brand", "price", "i_target_age_years"])
    hist_items = hist_tx.join(item_sub, on="item_id", how="left")
    
    # Precomputations cho HHI & Affinities
    u_tx_counts = hist_tx.group_by("customer_id").agg(pl.len().cast(pl.Float32).alias("u_total_tx"))
    ui_counts = hist_tx.group_by(["customer_id", "item_id"]).agg(pl.len().alias("ui_purchases"))
    
    user_loc = (
        hist_tx.group_by(["customer_id", "location"]).agg(pl.len().alias("loc_qty"))
        .sort(["customer_id", "loc_qty"], descending=[False, True])
        .group_by("customer_id").head(1).select(["customer_id", "location"])
    )
    
    loc_hhi = (
        hist_tx.group_by(["customer_id", "location"]).agg(pl.len().alias("loc_qty"))
        .join(u_tx_counts, on="customer_id")
        .with_columns((pl.col("loc_qty") / pl.col("u_total_tx")).alias("share"))
        .group_by("customer_id").agg([(pl.col("share") * pl.col("share")).sum().alias("u_loc_hhi")])
    )
    
    u_cat_affinity = (
        hist_items.group_by(["customer_id", "category_l1"]).agg(pl.col("quantity").sum().cast(pl.Float32).alias("u_cat_purchases"))
        .join(u_tx_counts, on="customer_id")
        .with_columns((pl.col("u_cat_purchases") / pl.col("u_total_tx")).alias("u_cat_share_of_wallet"))
    ).drop("u_total_tx")
    
    u_cat_hhi = (
        u_cat_affinity.group_by("customer_id").agg([
            (pl.col("u_cat_share_of_wallet") * pl.col("u_cat_share_of_wallet")).sum().alias("u_cat_hhi"),
            pl.col("category_l1").n_unique().alias("unique_cats")
        ])
    )
    
    u_brand_affinity = (
        hist_items.group_by(["customer_id", "brand"]).agg(pl.col("quantity").sum().cast(pl.Float32).alias("u_brand_purchases"))
        .join(u_tx_counts, on="customer_id")
        .with_columns((pl.col("u_brand_purchases") / pl.col("u_total_tx")).alias("u_brand_share_of_wallet"))
    ).drop("u_total_tx")
    
    u_brand_hhi = (
        u_brand_affinity.group_by("customer_id").agg([(pl.col("u_brand_share_of_wallet") * pl.col("u_brand_share_of_wallet")).sum().alias("u_brand_hhi")])
    )
    
    # Child Age Estimates
    tx_with_age = hist_items.filter(pl.col("i_target_age_years").is_not_null())
    user_child_age = (
        tx_with_age.sort(["customer_id", "event_ts"], descending=[False, True])
        .group_by("customer_id").head(1)
        .with_columns(
            (pl.col("i_target_age_years") + (pl.lit(max_ts) - pl.col("event_ts")).dt.total_days() / 365.0).cast(pl.Float32).alias("u_child_age_estimate")
        ).select(["customer_id", "u_child_age_estimate"])
    )
    
    # Advanced Basket & Temporal
    u_basket = hist_tx.group_by(["customer_id", "bill_id"]).agg(pl.col("item_id").n_unique().alias("bill_items")).group_by("customer_id").agg(pl.col("bill_items").mean().cast(pl.Float32).alias("u_avg_items_per_bill"))
    i_basket = hist_tx.group_by(["item_id", "bill_id"]).agg(pl.len().alias("qty")).group_by("item_id").agg(pl.col("qty").mean().cast(pl.Float32).alias("i_avg_items_in_its_bills"))
    
    lazy_time = hist_tx.with_columns([
        pl.col("event_ts").dt.hour().alias("hour"),
        (pl.col("event_ts").dt.weekday() >= 6).cast(pl.Float32).alias("is_weekend")
    ])
    u_time = lazy_time.group_by("customer_id").agg([
        pl.col("is_weekend").mean().cast(pl.Float32).alias("u_weekend_ratio"),
        pl.col("hour").mean().cast(pl.Float32).alias("u_avg_hour")
    ])
    i_time = lazy_time.group_by("item_id").agg([
        pl.col("is_weekend").mean().cast(pl.Float32).alias("i_weekend_ratio"),
        pl.col("hour").mean().cast(pl.Float32).alias("i_avg_hour")
    ])
    
    # Replenishment gap
    i_gaps = hist_tx.select(["customer_id", "item_id", "event_ts"]).sort(["customer_id", "item_id", "event_ts"]).with_columns(
        (pl.col("event_ts") - pl.col("event_ts").shift(1).over(["customer_id", "item_id"])).dt.total_days().alias("gap")
    ).filter(pl.col("gap").is_not_null() & (pl.col("gap") > 1))
    item_median_gap = i_gaps.group_by("item_id").agg(pl.col("gap").median().cast(pl.Float32).alias("i_median_replenish_gap"))

    # Champion Profile Assembler
    profile = (
        hist_items.group_by("customer_id")
        .agg([
            pl.len().alias("u_total_tx"),
            pl.col("quantity").sum().alias("u_total_qty"),
            (pl.col("quantity") * pl.col("price")).sum().alias("u_total_spend"),
            pl.col("item_id").n_unique().alias("u_unique_items"),
            (pl.lit(max_ts) - pl.col("event_ts").min()).dt.total_days().alias("u_tenure_days"),
            (pl.lit(max_ts) - pl.col("event_ts").max()).dt.total_days().alias("u_recency_days"),
        ])
        .join(loc_hhi, on="customer_id", how="left")
        .join(u_cat_hhi, on="customer_id", how="left")
        .join(u_brand_hhi, on="customer_id", how="left")
        .with_columns([
            (pl.col("u_total_qty") / pl.col("u_total_tx")).alias("u_avg_basket_size"),
            (pl.col("u_total_spend") / pl.col("u_total_tx")).alias("u_avg_order_value"),
            (pl.col("u_total_tx") / (pl.col("u_tenure_days") + 1.0)).alias("u_tx_velocity"),
            pl.col("u_loc_hhi").fill_null(1.0), pl.col("u_cat_hhi").fill_null(1.0),
            pl.col("u_brand_hhi").fill_null(1.0), pl.col("u_recency_days").fill_null(999.0)
        ])
        .with_columns(
            pl.when(pl.col("u_recency_days") >= 90).then(pl.lit("Dormant"))
            .when(pl.col("u_tenure_days") <= 60).then(pl.lit("New"))
            .when((pl.col("u_cat_hhi") >= 0.7) & (pl.col("u_total_tx") >= 3)).then(pl.lit("Habitual"))
            .when(pl.col("unique_cats") >= 4).then(pl.lit("Explorer"))
            .otherwise(pl.lit("Standard"))
            .alias("archetype")
        )
    )
    archetypes_df = target_users.join(profile, on="customer_id", how="left").with_columns(pl.col("archetype").fill_null("Dormant"))
    
    # Item Sales Windows Features
    i_sales_all = hist_tx.group_by("item_id").agg(pl.len().alias("i_sales_all"))
    i_sales_30d = hist_tx.filter(pl.col("event_ts") >= t_30).group_by("item_id").agg(pl.len().alias("i_sales_30d"))
    i_sales_60d = hist_tx.filter((pl.col("event_ts") >= t_60) & (pl.col("event_ts") < t_30)).group_by("item_id").agg(pl.len().alias("i_sales_60_to_30d"))
    i_sales_90d = hist_tx.filter((pl.col("event_ts") >= t_90) & (pl.col("event_ts") < t_60)).group_by("item_id").agg(pl.len().alias("i_sales_90_to_60d"))
    i_repeat_stats = ui_counts.group_by("item_id").agg([
        pl.col("ui_purchases").sum().alias("total_item_purchases"),
        pl.col("ui_purchases").filter(pl.col("ui_purchases") > 1).sum().alias("repeat_item_purchases")
    ]).with_columns((pl.col("repeat_item_purchases") / (pl.col("total_item_purchases") + 1e-6)).alias("item_repeat_propensity"))
    
    i_launch = hist_tx.group_by("item_id").agg([(pl.lit(max_ts) - pl.col("event_ts").min()).dt.total_days().alias("item_launch_age_days")])
    
    cat_sales_30d = hist_tx.filter(pl.col("event_ts") >= t_30).join(items, on="item_id", how="left").group_by("category_l1").agg(pl.len().alias("cat_sales_30d"))
    cat_sales_60d = hist_tx.filter((pl.col("event_ts") >= t_60) & (pl.col("event_ts") < t_30)).join(items, on="item_id", how="left").group_by("category_l1").agg(pl.len().alias("cat_sales_60d"))
    cat_trend_feat = cat_sales_30d.join(cat_sales_60d, on="category_l1", how="left").fill_null(0).with_columns((pl.col("cat_sales_30d") - pl.col("cat_sales_60d")).alias("cat_momentum"))
    
    cat_habitual = (
        ui_counts.join(items.select(["item_id", "category_l1"]), on="item_id", how="left")
        .group_by("category_l1")
        .agg([
            pl.col("ui_purchases").sum().alias("total_purchases"),
            pl.col("ui_purchases").filter(pl.col("ui_purchases") > 1).sum().alias("repurchases")
        ])
        .with_columns((pl.col("repurchases") / (pl.col("total_purchases") + 1e-6)).cast(pl.Float32).alias("cat_habitual_score"))
        .select(["category_l1", "cat_habitual_score"])
    )

    item_features_champ = (
        items.select(["item_id", "category_l1", "brand", "price", "i_target_age_years"])
        .join(i_sales_all, on="item_id", how="left")
        .join(i_sales_30d, on="item_id", how="left")
        .join(i_sales_60d, on="item_id", how="left")
        .join(i_sales_90d, on="item_id", how="left")
        .join(cat_trend_feat, on="category_l1", how="left")
        .join(i_repeat_stats, on="item_id", how="left")
        .join(i_launch, on="item_id", how="left")
        .fill_null(0)
        .with_columns([
            (pl.col("i_sales_30d") - pl.col("i_sales_60_to_30d")).alias("i_momentum_30d"),
            (pl.col("i_sales_60_to_30d") - pl.col("i_sales_90_to_60d")).alias("i_momentum_60d")
        ])
    )

    # User-Item Interaction Table
    ui_features_champ = (
        hist_tx.group_by(["customer_id", "item_id"]).agg([
            pl.len().alias("ui_purchase_count"),
            pl.col("quantity").sum().alias("ui_total_qty"),
            (pl.lit(max_ts) - pl.col("event_ts").max()).dt.total_days().alias("ui_days_since_last"),
            (pl.lit(max_ts) - pl.col("event_ts").min()).dt.total_days().alias("ui_days_since_first")
        ])
        .join(user_child_age, on="customer_id", how="left")
        .join(items.select(["item_id", "i_target_age_years"]), on="item_id", how="left")
        .with_columns([
            (pl.col("ui_total_qty") / pl.col("ui_purchase_count")).alias("ui_avg_qty_per_order"),
            (pl.col("ui_purchase_count") / (pl.col("ui_days_since_first") + 1.0)).alias("ui_purchase_velocity"),
            (pl.col("ui_days_since_last") > 22.0).cast(pl.Float32).alias("ui_replenishment_due"),
            pl.when(pl.col("u_child_age_estimate").is_not_null() & pl.col("i_target_age_years").is_not_null()).then(
                pl.col("u_child_age_estimate") - pl.col("i_target_age_years")
            ).otherwise(0.0).alias("ui_age_delta"),
            (-pl.col("ui_days_since_last") / 15.0).exp().cast(pl.Float32).alias("ui_recency_weight")
        ])
    )

    paths = {
        "archetypes": CACHE_DIR / "user_archetypes.parquet",
        "user_loc": CACHE_DIR / "profile_user_loc.parquet",
        "user_feat_champ": CACHE_DIR / "user_feat_champ.parquet",
        "item_feat_champ": CACHE_DIR / "item_feat_champ.parquet",
        "ui_feat_champ": CACHE_DIR / "ui_feat_champ.parquet",
        "u_cat_affinity": CACHE_DIR / "u_cat_affinity.parquet",
        "u_brand_affinity": CACHE_DIR / "u_brand_affinity.parquet",
        "u_basket": CACHE_DIR / "u_basket.parquet",
        "i_basket": CACHE_DIR / "i_basket.parquet",
        "u_time": CACHE_DIR / "u_time.parquet",
        "i_time": CACHE_DIR / "i_time.parquet",
        "item_median_gap": CACHE_DIR / "item_median_gap.parquet",
        "cat_habitual": CACHE_DIR / "cat_habitual.parquet"
    }
    
    archetypes_df.write_parquet(paths["archetypes"])
    user_loc.write_parquet(paths["user_loc"])
    profile.write_parquet(paths["user_feat_champ"])
    item_features_champ.write_parquet(paths["item_feat_champ"])
    ui_features_champ.write_parquet(paths["ui_feat_champ"])
    u_cat_affinity.write_parquet(paths["u_cat_affinity"])
    u_brand_affinity.write_parquet(paths["u_brand_affinity"])
    u_basket.write_parquet(paths["u_basket"])
    i_basket.write_parquet(paths["i_basket"])
    u_time.write_parquet(paths["u_time"])
    i_time.write_parquet(paths["i_time"])
    item_median_gap.write_parquet(paths["item_median_gap"])
    cat_habitual.write_parquet(paths["cat_habitual"])

    del hist_items, loc_hhi, u_cat_hhi, u_brand_hhi, user_child_age, tx_with_age, u_basket, i_basket, lazy_time, u_time, i_time, i_gaps, item_median_gap, i_sales_all, i_sales_30d, i_sales_60d, i_sales_90d, i_repeat_stats, i_launch, cat_sales_30d, cat_sales_60d, cat_trend_feat, item_features_champ, ui_features_champ, u_cat_affinity, u_brand_affinity, profile, archetypes_df; gc.collect()
    return paths

# ==============================================================================
# COMPACT REFERENCE MAP CHANNELS 
# ==============================================================================
def channel_history(hist_tx: pl.DataFrame) -> pl.DataFrame:
    print_status("Running Channel A: Purchase History...")
    return (
        hist_tx.group_by(["customer_id", "item_id"])
        .agg([pl.len().alias("purchase_count"), pl.col("event_ts").max().alias("last_ts")])
        .sort(["customer_id", "last_ts", "purchase_count"], descending=[False, True, True])
        .with_columns(pl.int_range(1, pl.len() + 1).over("customer_id").cast(pl.Int64).alias("rank"))
        .select(["customer_id", "item_id", "rank"])
    )

def channel_local_popular_map(hist_tx: pl.DataFrame, top_k: int = 500) -> pl.DataFrame:
    recent_tx = hist_tx.filter(pl.col("event_ts") >= hist_tx["event_ts"].max() - pl.duration(days=60))
    return (
        recent_tx.group_by(["location", "item_id"]).agg(pl.len().alias("qty"))
        .sort(["location", "qty"], descending=[False, True]).group_by("location").head(top_k)
        .with_columns(pl.int_range(1, pl.len() + 1).over("location").cast(pl.Int64).alias("rank"))
        .select(["location", "item_id", "rank"])
    )

def train_cf_latent(hist_tx: pl.DataFrame, svd_components: int = 100) -> tuple:
    print_status("Training Global Latent CF Matrices Natively via SciPy...")
    max_ts = hist_tx["event_ts"].max()
    tx_weighted = (
        hist_tx
        .with_columns(((pl.lit(max_ts) - pl.col("event_ts")).dt.total_days() / 30.0).alias("months_ago"))
        .with_columns((pl.col("quantity") * pl.lit(0.70).pow(pl.col("months_ago"))).cast(pl.Float32).alias("weight"))
        .group_by(["customer_id", "item_id"]).agg(pl.col("weight").sum().alias("weight"))
    )
    
    u_map, i_map = tx_weighted["customer_id"].unique(), hist_tx["item_id"].unique()
    u_df = pl.DataFrame({"customer_id": u_map, "u_idx": np.arange(len(u_map), dtype=np.int32)})
    i_df = pl.DataFrame({"item_id": i_map, "i_idx": np.arange(len(i_map), dtype=np.int32)})
    hybrid_indexed = tx_weighted.join(u_df, on="customer_id", how="inner").join(i_df, on="item_id", how="inner")
    
    rows, cols, data = hybrid_indexed["u_idx"].to_numpy(), hybrid_indexed["i_idx"].to_numpy(), hybrid_indexed["weight"].to_numpy()
    mtx = csr_matrix((data, (rows, cols)), shape=(len(u_map), len(i_map))).astype(np.float32)
    u2idx = dict(zip(u_df["customer_id"], u_df["u_idx"]))
    idx2i = i_map.to_list()
    i_arr = np.array(idx2i)
    
    U, S, Vt = svds(mtx, k=svd_components)
    sort_idx = np.argsort(S)[::-1]
    U = U[:, sort_idx]
    S = S[sort_idx]
    Vt = Vt[sort_idx, :]
    
    u_emb = (U * S).astype(np.float32)
    i_emb = Vt.T.astype(np.float32)
    
    col_norms = np.sqrt(np.array(mtx.power(2).sum(axis=0))).flatten()
    col_norms[col_norms == 0] = 1.0
    norm_m = mtx @ sp.diags(1.0 / col_norms)
    
    i2i_sim = (norm_m.T.dot(norm_m)).astype(np.float32)
    i2i_sim.setdiag(0)
    i2i_sim = i2i_sim.tocsr()
    i2i_sim.data[i2i_sim.data < 0.05] = 0.0
    i2i_sim.eliminate_zeros()
    
    del tx_weighted, u_df, i_df, hybrid_indexed, norm_m, col_norms; gc.collect()
    return u_emb, i_emb, i2i_sim, mtx, u2idx, i_arr

def channel_category_popular_map(hist_tx: pl.DataFrame, items: pl.DataFrame, bestsellers_per_category: int = 80) -> pl.DataFrame:
    item_cat = items.select(["item_id", "category_l1"])
    recent_tx = hist_tx.filter(pl.col("event_ts") >= hist_tx["event_ts"].max() - pl.duration(days=45))
    return (
        recent_tx.join(item_cat, on="item_id", how="left").filter(pl.col("category_l1") != "Unknown")
        .group_by(["category_l1", "item_id"]).agg(pl.len().alias("qty"))
        .sort(["category_l1", "qty"], descending=[False, True]).group_by("category_l1").head(bestsellers_per_category)
        .with_columns(pl.int_range(1, pl.len() + 1).over("category_l1").cast(pl.Int64).alias("item_rank"))
        .select(["category_l1", "item_id", "item_rank"])
    )

def channel_brand_popular_map(hist_tx: pl.DataFrame, items: pl.DataFrame, bestsellers_per_brand: int = 50) -> pl.DataFrame:
    item_brand = items.select(["item_id", "brand"])
    recent_tx = hist_tx.filter(pl.col("event_ts") >= hist_tx["event_ts"].max() - pl.duration(days=60))
    return (
        recent_tx.join(item_brand, on="item_id", how="left").filter((pl.col("brand") != "Unknown") & (pl.col("brand") != "Không xác định"))
        .group_by(["brand", "item_id"]).agg(pl.len().alias("qty"))
        .sort(["brand", "qty"], descending=[False, True]).group_by("brand").head(bestsellers_per_brand)
        .with_columns(pl.int_range(1, pl.len() + 1).over("brand").cast(pl.Int64).alias("item_rank"))
        .select(["brand", "item_id", "item_rank"])
    )

def channel_global_popular_map(hist_tx: pl.DataFrame, global_k: int = 300) -> pl.DataFrame:
    recent = hist_tx.filter(pl.col("event_ts") >= hist_tx["event_ts"].max() - pl.duration(days=30))
    return (
        recent.group_by("item_id").agg(pl.len().alias("qty"))
        .sort("qty", descending=True).head(global_k)
        .with_columns(pl.int_range(1, pl.len() + 1).cast(pl.Int64).alias("rank"))
        .select(["item_id", "rank"])
    )

def channel_category_trending_map(hist_tx: pl.DataFrame, items: pl.DataFrame, trending: int = 50) -> pl.DataFrame:
    item_cat = items.select(["item_id", "category_l1"])
    max_ts = hist_tx["event_ts"].max()
    t_recent, t_prior = max_ts - pl.duration(days=30), max_ts - pl.duration(days=60)
    recent_sales = hist_tx.filter(pl.col("event_ts") >= t_recent).group_by("item_id").agg(pl.len().alias("qty_recent"))
    prior_sales = hist_tx.filter((pl.col("event_ts") >= t_prior) & (pl.col("event_ts") < t_recent)).group_by("item_id").agg(pl.len().alias("qty_recent"))
    
    momentum = (
        recent_sales.join(prior_sales, on="item_id", how="full").fill_null(0.0)
        .with_columns((pl.col("qty_recent") - pl.col("qty_prior")).alias("momentum"))
        .join(item_cat, on="item_id", how="left").filter(pl.col("category_l1") != "Unknown")
    )
    return (
        momentum.sort(["category_l1", "momentum"], descending=[False, True]).group_by("category_l1").head(trending)
        .with_columns(pl.int_range(1, pl.len() + 1).over("category_l1").cast(pl.Int64).alias("item_rank"))
        .select(["category_l1", "item_id", "item_rank"])
    )

def channel_s4_monthly_pop_map(hist_tx: pl.DataFrame, n_pop: int = 100) -> pl.DataFrame:
    max_ts = hist_tx["event_ts"].max()
    cutoff_1m = max_ts - pl.duration(days=30)
    return (
        hist_tx.filter(pl.col("event_ts") >= cutoff_1m)
        .group_by("item_id").agg(pl.len().alias("sales"))
        .sort("sales", descending=True).head(n_pop)
        .with_row_index("rank", offset=1).with_columns(pl.col("rank").cast(pl.Int64))
        .select(["item_id", "rank"])
    )

def channel_s5_monthly_trend_map(hist_tx: pl.DataFrame, n_trend: int = 20) -> pl.DataFrame:
    max_ts = hist_tx["event_ts"].max()
    t_m1 = max_ts - pl.duration(days=30)
    t_m2 = max_ts - pl.duration(days=60)
    
    sales_m1 = hist_tx.filter(pl.col("event_ts") >= t_m1).group_by("item_id").agg(pl.len().alias("sales_m1"))
    sales_m2 = hist_tx.filter((pl.col("event_ts") >= t_m2) & (pl.col("event_ts") < t_m1)).group_by("item_id").agg(pl.len().alias("sales_m2"))
    
    return (
        sales_m1.join(sales_m2, on="item_id", how="left").fill_null(0)
        .filter(pl.col("sales_m1") >= 10)
        .with_columns((pl.col("sales_m1") / (pl.col("sales_m2") + 1.0)).alias("trend_ratio"))
        .sort("trend_ratio", descending=True).head(n_trend)
        .with_row_index("rank", offset=1).with_columns(pl.col("rank").cast(pl.Int64))
        .select(["item_id", "rank"])
    )

def channel_s1_history_recent(hist_tx: pl.DataFrame, hist_events: pl.DataFrame, n_history: int = 200) -> pl.DataFrame:
    max_ts = hist_tx["event_ts"].max()
    cutoff_2m = max_ts - pl.duration(days=60)
    tx_sub = hist_tx.filter(pl.col("event_ts") >= cutoff_2m).select(["customer_id", "item_id", "event_ts"])
    ev_sub = hist_events.filter(pl.col("event_ts") >= cutoff_2m).select(["customer_id", "item_id", "event_ts"])
    combined = pl.concat([tx_sub, ev_sub])
    return (
        combined.sort(["customer_id", "event_ts"], descending=[False, True])
        .unique(subset=["customer_id", "item_id"], keep="first")
        .group_by("customer_id").head(n_history)
        .with_columns(pl.int_range(1, pl.len() + 1).over("customer_id").cast(pl.Int64).alias("rank"))
        .select(["customer_id", "item_id", "rank"])
    )

def train_s2_als(hist_tx: pl.DataFrame) -> tuple:
    print_status("Training Global Advanced ALS CF Model...")
    unique_users = hist_tx.select("customer_id").unique().with_row_index("u_idx")
    unique_items = hist_tx.select("item_id").unique().with_row_index("i_idx")
    interactions = (
        hist_tx.group_by(["customer_id", "item_id"]).agg(pl.len().alias("weight"))
        .join(unique_users, on="customer_id").join(unique_items, on="item_id")
    )
    user_item_matrix = sp.csr_matrix(
        (interactions["weight"].to_numpy(), (interactions["u_idx"].to_numpy(), interactions["i_idx"].to_numpy())),
        shape=(unique_users.height, unique_items.height)
    ).astype(np.float32)
    
    model = implicit.als.AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01, random_state=42)
    model.fit(user_item_matrix)
    als_u2idx = dict(zip(unique_users["customer_id"], unique_users["u_idx"]))
    als_i_arr = unique_items.sort("i_idx")["item_id"].to_numpy()
    del unique_users, unique_items, interactions; gc.collect()
    return model, user_item_matrix, als_u2idx, als_i_arr

def channel_s6_full_history(hist_tx: pl.DataFrame, n_full_history: int = 300) -> pl.DataFrame:
    print_status("Running Channel S6: Full History Freq Count...")
    return (
        hist_tx.group_by(["customer_id", "item_id"]).agg(pl.len().alias("ui_purchases"))
        .sort(["customer_id", "ui_purchases"], descending=[False, True])
        .group_by("customer_id").head(n_full_history)
        .with_columns(pl.int_range(1, pl.len() + 1).over("customer_id").cast(pl.Int64).alias("rank"))
        .select(["customer_id", "item_id", "rank"])
    )

# ==============================================================================
# DISK-BASED STREAMING CHUNK FUSION ENGINE (STREAMING MAPPING)
# ==============================================================================
def evaluate_and_fuse(channel_files: dict, ref_maps: dict, profile_paths: dict, latent_models: dict, max_final_candidates: int = 150, label_tx: pl.DataFrame = None, model: lgb.Booster = None) -> pl.DataFrame:
    print_status("Fusing candidates dynamically and assembling raw feature matrices natively...")
    
    unique_users = pl.read_parquet(profile_paths["archetypes"]).select("customer_id").unique()["customer_id"].to_list()
    u_emb, i_emb, i2i_sim, mtx, u2idx, i_arr = latent_models["cf_latent"]
    als_model, als_matrix, als_u2idx, als_i_arr = latent_models["als"]
    k_svd = latent_models["k_svd"]
    
    print_status("Pre-loading lightweight lookup bases into memory space...")
    full_archetypes = pl.read_parquet(profile_paths["archetypes"])
    full_user_feat = pl.read_parquet(profile_paths["user_feat_champ"])
    full_item_feat = pl.read_parquet(profile_paths["item_feat_champ"])
    full_ui_feat = pl.read_parquet(profile_paths["ui_feat_champ"])
    full_user_loc = pl.read_parquet(profile_paths["user_loc"])
    full_u_cat = pl.read_parquet(profile_paths["u_cat_affinity"])
    full_u_brand = pl.read_parquet(profile_paths["u_brand_affinity"])
    full_u_basket = pl.read_parquet(profile_paths["u_basket"])
    full_i_basket = pl.read_parquet(profile_paths["i_basket"])
    full_u_time = pl.read_parquet(profile_paths["u_time"])
    full_i_time = pl.read_parquet(profile_paths["i_time"])
    full_item_gap = pl.read_parquet(profile_paths["item_median_gap"])
    full_cat_habitual = pl.read_parquet(profile_paths["cat_habitual"])
    
    full_user_base_features = pl.read_parquet(profile_paths["user_features"])
    full_item_base_features = pl.read_parquet(profile_paths["item_features"])
    
    if label_tx is not None:
        unique_users = label_tx.select("customer_id").unique()["customer_id"].to_list()
        print_status(f"[TRAIN MODE] Active learning subset restricted to exactly {len(unique_users):,} target users.")
    else:
        unique_users = full_archetypes.select("customer_id").unique()["customer_id"].to_list()
        print_status(f"[PREDICT MODE] Running full-scale inference partition for {len(unique_users):,} unique users.")

    loaded_channels = {c_name: pl.read_parquet(f_path) for c_name, f_path in channel_files.items()}

    chunk_size = 10000
    num_chunks = int(np.ceil(len(unique_users) / chunk_size))
    
    weight_map = {
        "Habitual": {"A_history": 5.0, "B_local": 1.0, "C_svd": 0.5, "D_i2i": 0.5, "E_cat": 2.0, "F_brand": 3.0, "G_global": 0.1, "H_trend": 0.5, "S1_hist_recent": 4.0, "S2_als": 1.0, "S3_cobuy": 1.0, "S4_month_pop": 0.2, "S5_month_trend": 0.5, "S6_full_hist": 4.5},
        "Explorer": {"A_history": 1.0, "B_local": 1.0, "C_svd": 4.0, "D_i2i": 4.0, "E_cat": 3.0, "F_brand": 1.0, "G_global": 1.0, "H_trend": 2.0, "S1_hist_recent": 1.5, "S2_als": 4.5, "S3_cobuy": 3.5, "S4_month_pop": 1.0, "S5_month_trend": 2.5, "S6_full_hist": 0.5},
        "Dormant":  {"A_history": 3.0, "B_local": 4.0, "C_svd": 0.5, "D_i2i": 0.5, "E_cat": 1.0, "F_brand": 2.0, "G_global": 5.0, "H_trend": 1.0, "S1_hist_recent": 2.0, "S2_als": 0.5, "S3_cobuy": 0.5, "S4_month_pop": 4.5, "S5_month_trend": 2.0, "S6_full_hist": 3.5},
        "New":      {"A_history": 0.5, "B_local": 3.0, "C_svd": 2.5, "D_i2i": 2.5, "E_cat": 3.0, "F_brand": 1.0, "G_global": 4.0, "H_trend": 4.0, "S1_hist_recent": 0.5, "S2_als": 3.0, "S3_cobuy": 2.0, "S4_month_pop": 4.5, "S5_month_trend": 4.5, "S6_full_hist": 0.1},
        "Standard": {"A_history": 2.0, "B_local": 2.0, "C_svd": 1.5, "D_i2i": 1.5, "E_cat": 2.0, "F_brand": 2.0, "G_global": 1.0, "H_trend": 1.5, "S1_hist_recent": 2.0, "S2_als": 2.0, "S3_cobuy": 2.0, "S4_month_pop": 1.5, "S5_month_trend": 2.0, "S6_full_hist": 2.0},
    }
    w_df = pl.DataFrame([{"archetype": a, "channel": c, "ch_weight": w} for a, cw in weight_map.items() for c, w in cw.items()])
    
    if label_tx is not None:
        output_path = CACHE_DIR / "lgbm_train_matrix.parquet"
        arrow_schema = pa.schema([
            ('customer_id', pa.int32()), 
            ('item_id', pa.large_string()),
            *[(f, pa.float32()) for f in LGBM_FEATURES],
            ('label', pa.int32())
        ])
    else:
        output_path = CACHE_DIR / "final_recommendations_nhap.parquet"
        arrow_schema = pa.schema([('customer_id', pa.int32()), ('item_id', pa.large_string())])
        
    writer = pq.ParquetWriter(str(output_path), arrow_schema, compression='SNAPPY')
    
    for idx_loop in range(0, len(unique_users), chunk_size):
        curr_chunk_idx = (idx_loop // chunk_size) + 1
        if curr_chunk_idx % 10 == 0 or curr_chunk_idx == 1 or curr_chunk_idx == num_chunks:
            print_status(f" -> Streaming Block Chunk {curr_chunk_idx}/{num_chunks}...")
            
        chunk_u = unique_users[idx_loop : idx_loop + chunk_size]
        chunk_u_df = pl.DataFrame({"customer_id": chunk_u}, schema={"customer_id": pl.Int32})
        
        arch_chunk = full_archetypes.join(chunk_u_df, on="customer_id", how="inner")
        user_feat_chunk = full_user_feat.join(chunk_u_df, on="customer_id", how="inner")
        chunk_cands_list = []
        
        for c_name, df_mem in loaded_channels.items():
            df_c = df_mem.join(chunk_u_df, on="customer_id", how="inner")
            if not df_c.is_empty():
                df_c = df_c.with_columns(pl.lit(c_name).alias("channel"))
                chunk_cands_list.append(df_c.select(["customer_id", "item_id", "rank", "channel"]))
                
        chunk_u_mapped = [u2idx[u] for u in chunk_u if u in u2idx]
        chunk_users_matched = [u for u in chunk_u if u in u2idx]
        
        if chunk_u_mapped:
            k_top = 150
            scores_svd_mat = u_emb[chunk_u_mapped] @ i_emb.T
            top_svd = np.argpartition(-scores_svd_mat, k_top, axis=1)[:, :k_top]
            row_idx = np.arange(top_svd.shape[0])[:, None]
            top_svd = top_svd[row_idx, np.argsort(-scores_svd_mat[row_idx, top_svd], axis=1)]
            
            df_c_svd = pl.DataFrame({
                "customer_id": np.repeat(chunk_users_matched, 150).astype(np.int32),
                "item_id": i_arr[top_svd.flatten()],
                "rank": np.tile(np.arange(1, 151, dtype=np.int32), len(chunk_users_matched)).astype(np.int64),
                "channel": ["C_svd"] * (len(chunk_users_matched) * 150)
            }).select([pl.col("customer_id").cast(pl.Int32), pl.col("item_id").cast(pl.Utf8), pl.col("rank").cast(pl.Int64), pl.col("channel").cast(pl.Utf8)])
            chunk_cands_list.append(df_c_svd)
            
            scores_i2i = mtx[chunk_u_mapped].dot(i2i_sim).toarray()
            top_i2i = np.argpartition(-scores_i2i, k_top, axis=1)[:, :k_top]
            row_idx_i2i = np.arange(top_i2i.shape[0])[:, None]
            top_i2i = top_i2i[row_idx_i2i, np.argsort(-scores_i2i[row_idx_i2i, top_i2i], axis=1)]
            mask = np.take_along_axis(scores_i2i, top_i2i, axis=1) > 0.0
            
            df_d_i2i = pl.DataFrame({
                "customer_id": np.repeat(chunk_users_matched, 150)[mask.flatten()].astype(np.int32),
                "item_id": i_arr[top_i2i.flatten()][mask.flatten()],
                "rank": np.tile(np.arange(1, 151, dtype=np.int32), len(chunk_users_matched))[mask.flatten()].astype(np.int64),
                "channel": ["D_i2i"] * np.sum(mask)
            }).select([pl.col("customer_id").cast(pl.Int32), pl.col("item_id").cast(pl.Utf8), pl.col("rank").cast(pl.Int64), pl.col("channel").cast(pl.Utf8)])
            chunk_cands_list.append(df_d_i2i)
            del scores_svd_mat, top_svd, scores_i2i, top_i2i, mask, df_c_svd, df_d_i2i; gc.collect()

        chunk_als_u = [als_u2idx[u] for u in chunk_u if u in als_u2idx]
        chunk_users_als = [u for u in chunk_u if u in als_u2idx]
        if chunk_als_u:
            ids, _ = als_model.recommend(chunk_als_u, als_matrix[chunk_als_u], N=150, filter_already_liked_items=False)
            df_s2_als = pl.DataFrame({
                "customer_id": np.repeat(chunk_users_als, 150).astype(np.int32),
                "item_id": als_i_arr[ids.flatten()],
                "rank": np.tile(np.arange(1, 151, dtype=np.int32), len(chunk_users_als)).astype(np.int64),
                "channel": ["S2_als"] * (len(chunk_users_als) * 150)
            }).select([pl.col("customer_id").cast(pl.Int32), pl.col("item_id").cast(pl.Utf8), pl.col("rank").cast(pl.Int64), pl.col("channel").cast(pl.Utf8)])
            chunk_cands_list.append(df_s2_als)
            del ids, df_s2_als; gc.collect()

        u_loc_c = full_user_loc.join(chunk_u_df, on="customer_id", how="inner")
        df_b = u_loc_c.join(ref_maps["B_local"], on="location", how="inner").drop("location")
        chunk_cands_list.append(df_b.with_columns(pl.lit("B_local").alias("channel")).select(["customer_id", "item_id", "rank", "channel"]))
        
        u_cat_c = full_u_cat.join(chunk_u_df, on="customer_id", how="inner")
        df_e = u_cat_c.join(ref_maps["E_cat"], on="category_l1", how="inner").drop("category_l1")
        df_e = df_e.with_columns((pl.col("item_id").cum_count().over("customer_id") + 1).cast(pl.Int64).alias("rank"))
        chunk_cands_list.append(df_e.with_columns(pl.lit("E_cat").alias("channel")).select(["customer_id", "item_id", "rank", "channel"]))
        
        u_brd_c = full_u_brand.join(chunk_u_df, on="customer_id", how="inner")
        df_f = u_brd_c.join(ref_maps["F_brand"], on="brand", how="inner").drop("brand")
        df_f = df_f.with_columns((pl.col("item_id").cum_count().over("customer_id") + 1).cast(pl.Int64).alias("rank"))
        chunk_cands_list.append(df_f.with_columns(pl.lit("F_brand").alias("channel")).select(["customer_id", "item_id", "rank", "channel"]))
        
        df_h = u_cat_c.join(ref_maps["H_trend"], on="category_l1", how="inner").drop("category_l1")
        df_h = df_h.with_columns((pl.col("item_id").cum_count().over("customer_id") + 1).cast(pl.Int64).alias("rank"))
        chunk_cands_list.append(df_h.with_columns(pl.lit("H_trend").alias("channel")).select(["customer_id", "item_id", "rank", "channel"]))
        
        df_g = chunk_u_df.join(ref_maps["G_global"], how="cross").with_columns(pl.lit("G_global").alias("channel")).select(["customer_id", "item_id", "rank", "channel"])
        df_s4 = chunk_u_df.join(ref_maps["S4_month_pop"], how="cross").with_columns(pl.lit("S4_month_pop").alias("channel")).select(["customer_id", "item_id", "rank", "channel"])
        df_s5 = chunk_u_df.join(ref_maps["S5_month_trend"], how="cross").with_columns(pl.lit("S5_month_trend").alias("channel")).select(["customer_id", "item_id", "rank", "channel"])
        
        chunk_cands_list.extend([df_g, df_s4, df_s5])

        stacked_chunk = pl.concat(chunk_cands_list)
        merged = stacked_chunk.join(arch_chunk.select(["customer_id", "archetype"]), on="customer_id", how="inner")
        
        candidate_pool = (
            merged.join(w_df, on=["archetype", "channel"], how="left")
            .with_columns(pl.col("ch_weight").fill_null(1.0))
            .with_columns((pl.col("ch_weight") / (pl.col("rank") + 60.0)).alias("score"))
            .group_by(["customer_id", "item_id", "archetype"])
            .agg(pl.col("score").sum().alias("rrf_score"))
            .sort(["customer_id", "rrf_score"], descending=[False, True])
            .with_columns(pl.col("item_id").cum_count().over("customer_id").cast(pl.Int64).alias("final_rank"))
            .filter(pl.col("final_rank") <= max_final_candidates)
            .drop("final_rank")
        )
        
        merged_filtered = merged.join(candidate_pool.select(["customer_id", "item_id"]), on=["customer_id", "item_id"], how="inner")
        
        if not merged_filtered.is_empty():
            channels_pivoted = merged_filtered.pivot(
                on="channel", index=["customer_id", "item_id"], values="rank", aggregate_function="first"
            )
            with_cols = []
            for col in CHANNELS:
                if col not in channels_pivoted.columns:
                    with_cols.append(pl.lit(999.0).alias(col).cast(pl.Float32))
                else:
                    with_cols.append(pl.col(col).cast(pl.Float32).fill_null(999.0))
            channels_pivoted = channels_pivoted.with_columns(with_cols)
        else:
            channels_pivoted = pl.DataFrame({"customer_id": pl.Series([], dtype=pl.Int32), "item_id": pl.Series([], dtype=pl.Utf8), **{col: pl.Series([], dtype=pl.Float32) for col in CHANNELS}})
            
        scored = candidate_pool.join(channels_pivoted, on=["customer_id", "item_id"], how="inner")
        
        # TÍNH TOÁN VÀ PHÂN PHỐI BỂ ĐẶC TRƯNG HÀNH VI CHO CHUNK HIỆN TẠI
        scored = (
            scored
            .join(full_user_loc.join(chunk_u_df, on="customer_id", how="inner"), on="customer_id", how="left")
            .join(user_feat_chunk, on="customer_id", how="inner")
            .join(full_user_base_features.join(chunk_u_df, on="customer_id", how="inner"), on="customer_id", how="inner")
            .join(full_item_feat, on="item_id", how="inner")
            .join(full_item_base_features, on="item_id", how="inner")
            .join(full_ui_feat.join(chunk_u_df, on="customer_id", how="inner"), on=["customer_id", "item_id"], how="left")
            .join(u_cat_c, on=["customer_id", "category_l1"], how="left")
            .join(u_brd_c, on=["customer_id", "brand"], how="left")
            .join(full_u_basket.join(chunk_u_df, on="customer_id", how="inner"), on="customer_id", how="left")
            .join(full_i_basket, on="item_id", how="left")
            .join(full_u_time.join(chunk_u_df, on="customer_id", how="inner"), on="customer_id", how="left")
            .join(full_i_time, on="item_id", how="left")
            .join(full_item_gap, on="item_id", how="left")
            .join(full_cat_habitual, on="category_l1", how="left")
        )
        
        fill_dict = {
            "ui_purchase_count": 0.0, "ui_total_qty": 0.0, "ui_days_since_last": 999.0,
            "ui_days_since_first": 999.0, "ui_avg_qty_per_order": 0.0, "ui_purchase_velocity": 0.0,
            "ui_replenishment_due": 0.0, "ui_recency_weight": 0.0, "u_cat_purchases": 0.0,
            "u_cat_share_of_wallet": 0.0, "u_brand_purchases": 0.0, "u_brand_share_of_wallet": 0.0,
            "u_child_age_estimate": -99.0, "u_avg_items_per_bill": 1.0, "i_avg_items_in_its_bills": 1.0,
            "u_weekend_ratio": 0.0, "u_avg_hour": 12.0, "i_weekend_ratio": 0.0, "i_avg_hour": 12.0,
            "i_median_replenish_gap": 30.0, "cat_habitual_score": 0.5,
            "u_avg_price": 0.0, "item_avg_price": 0.0, "i_total_sales": 0.0
        }
        scored = scored.with_columns([
            pl.col(c).fill_null(v).cast(pl.Float32) for c, v in fill_dict.items() if c in scored.columns
        ])
        
        # Sinh các đặc trưng tương tác chéo phi tuyến hóa (Cross Features)
        scored = scored.with_columns([
            (pl.col("u_cat_share_of_wallet") * pl.col("i_momentum_30d")).alias("cross_cat_momentum"),
            (pl.col("u_brand_share_of_wallet") * pl.col("i_momentum_30d")).alias("cross_brand_momentum"),
            (pl.col("price") / (pl.col("u_avg_order_value") + 1.0)).alias("cross_price_ratio"),
            pl.when(pl.col("ui_purchase_count") > 0).then(pl.col("cat_habitual_score")).otherwise(1.0 - pl.col("cat_habitual_score")).alias("ui_habitual_match"),
            (pl.col("u_avg_items_per_bill") - pl.col("i_avg_items_in_its_bills")).abs().alias("basket_size_mismatch"),
            (pl.col("u_weekend_ratio") * pl.col("i_weekend_ratio")).alias("weekend_shopper_match"),
            (pl.col("ui_days_since_last") - pl.col("i_median_replenish_gap")).alias("replenishment_overdue_days")
        ])
        
        # Đồng bộ hóa đặc trưng SVD tương đồng gốc
        svd_dot_expr = pl.lit(0.0)
        for idx in range(k_svd):
            svd_dot_expr = svd_dot_expr + (pl.col(f"u_svd_{idx}").fill_null(0.0) * pl.col(f"c_svd_{idx}").fill_null(0.0))
            
        scored = scored.with_columns([
            svd_dot_expr.alias("svd_similarity"),
            (pl.col("item_avg_price") / (pl.col("u_avg_price") + 1e-6)).alias("price_ratio"),
            svd_dot_expr.alias("svd_score") 
        ])
        
        # Lấp đầy an toàn các trường null còn lại trong toàn bộ LGBM_FEATURES
        scored = scored.with_columns([
            pl.col(f).fill_null(0.0).cast(pl.Float32) for f in LGBM_FEATURES if f not in fill_dict
        ])
        
        if label_tx is not None:
            final_chunk = (
                scored.join(label_tx.select(["customer_id", "item_id"]).with_columns(pl.lit(1).alias("label")), on=["customer_id", "item_id"], how="left")
                .with_columns(pl.col("label").fill_null(0).cast(pl.Int32))
                .select(["customer_id", "item_id"] + LGBM_FEATURES + ["label"])
                .with_columns([
                    pl.col("customer_id").cast(pl.Int32), pl.col("item_id").cast(pl.Utf8),
                    *[pl.col(f).cast(pl.Float32) for f in LGBM_FEATURES],
                    pl.col("label").cast(pl.Int32)
                ])
            )
            writer.write_table(final_chunk.to_arrow())
        else:
            X_chunk = scored.select(LGBM_FEATURES).to_numpy()
            scores_lgbm = model.predict(X_chunk)
            
            final_chunk = (
                scored.with_columns(pl.Series("preds_lgbm", scores_lgbm, dtype=pl.Float32))
                .sort(["customer_id", "preds_lgbm"], descending=[False, True])
                .with_columns((pl.col("item_id").cum_count().over("customer_id") + 1).cast(pl.Int64).alias("final_rank"))
                .filter(pl.col("final_rank") <= 10)
                .select(["customer_id", "item_id"])
                .with_columns([pl.col("customer_id").cast(pl.Int32), pl.col("item_id").cast(pl.Utf8)])
            )
            writer.write_table(final_chunk.to_arrow())
            
        del stacked_chunk, merged, scored, final_chunk, arch_chunk, user_feat_chunk, u_loc_c, u_cat_c, u_brd_c, df_g, df_s4, df_s5, chunk_u_mapped, chunk_users_matched, chunk_u_df, channels_pivoted, candidate_pool, merged_filtered; gc.collect()

    writer.close()
    
    print_status("Streaming block checkpoint completed successfully. Re-loading parsed collection...")
    final = pl.read_parquet(output_path)
    del full_archetypes, full_user_feat, full_item_feat, full_ui_feat, full_user_loc, full_u_cat, full_u_brand, full_u_basket, full_i_basket, full_u_time, full_i_time, full_item_gap, full_cat_habitual, full_user_base_features, full_item_base_features, loaded_channels; gc.collect()
    return final

# ==============================================================================
# PIPELINE EXECUTION ENGINE (TWO-STAGE ML RANKER INTERFACING)
# ==============================================================================
def run_pipeline():
    print_status("--- Starting Full Feature-Integrated PIR Pipeline v2 (Two-Stage ML Scale) ---")
    
    items = load_items()
    hist_tx, target_users = load_history_data()
    
    if USER_LIMIT > 0:
        print_status(f"[SMOKE TEST ACTIVATED] Restricting entire pipeline scale down to exactly {USER_LIMIT} users.")
        target_users = target_users.head(USER_LIMIT)
        hist_tx = hist_tx.filter(pl.col("customer_id").is_in(target_users["customer_id"].to_list()))
        
    hist_events = load_event_history(target_users)
    
    # --------------------------------------------------------------------------
    # STAGE 2 - PHASE 1: TRÍCH XUẤT ĐẶC TRƯNG HỌC MÔ HÌNH (MONTHS 1-11 -> MONTH 12)
    # --------------------------------------------------------------------------
    print_status("[PIPELINE STAGE 2] Extracting training dataset bounds (Months 1-11 Features -> Month 12 Targets)...")
    label_tx_snapshot = hist_tx.filter((pl.col("event_ts") >= datetime(2025, 12, 1)) & (pl.col("event_ts") < datetime(2026, 1, 1)))
    train_tx_snapshot = hist_tx.filter(pl.col("event_ts") < datetime(2025, 12, 1))
    train_events_snapshot = hist_events.filter(pl.col("event_ts") < datetime(2025, 12, 1))
    
    p_uf, p_if, k_svd = precompute_advanced_features(train_tx_snapshot, items)
    profile_paths = compute_user_archetypes_and_profiles(train_tx_snapshot, items, target_users)
    profile_paths["user_features"] = p_uf
    profile_paths["item_features"] = p_if
    
    ref_maps = {
        "B_local": channel_local_popular_map(train_tx_snapshot, top_k=500),
        "E_cat": channel_category_popular_map(train_tx_snapshot, items, bestsellers_per_category=80),
        "F_brand": channel_brand_popular_map(train_tx_snapshot, items, bestsellers_per_brand=50),
        "G_global": channel_global_popular_map(train_tx_snapshot, global_k=300),
        "H_trend": channel_category_trending_map(train_tx_snapshot, items, trending=50),
        "S4_month_pop": channel_s4_monthly_pop_map(train_tx_snapshot, n_pop=100),
        "S5_month_trend": channel_s5_monthly_trend_map(train_tx_snapshot, n_trend=20)
    }
    
    latent_models = {"cf_latent": train_cf_latent(train_tx_snapshot), "als": train_s2_als(train_tx_snapshot), "k_svd": k_svd}
    
    channel_files = {}
    df_a = channel_history(train_tx_snapshot)
    p_a = CACHE_DIR / "ch_A_history.parquet"; df_a.write_parquet(p_a); channel_files["A_history"] = p_a
    df_s1 = channel_s1_history_recent(train_tx_snapshot, train_events_snapshot, n_history=200)
    p_s1 = CACHE_DIR / "ch_S1_hist_recent.parquet"; df_s1.write_parquet(p_s1); channel_files["S1_hist_recent"] = p_s1
    df_s6 = channel_s6_full_history(train_tx_snapshot, n_full_history=300)
    p_s6 = CACHE_DIR / "ch_S6_full_hist.parquet"; df_s6.write_parquet(p_s6); channel_files["S6_full_hist"] = p_s6
    del df_a, df_s1, df_s6, train_tx_snapshot, train_events_snapshot; gc.collect()
    
    train_matrix_df = evaluate_and_fuse(channel_files, ref_maps, profile_paths, latent_models, max_final_candidates=150, label_tx=label_tx_snapshot)
    
    # --------------------------------------------------------------------------
    # STAGE 2 - PHASE 2: LUYỆN MÔ HÌNH LIGHTGBM LAMBDARANK
    # --------------------------------------------------------------------------
    print_status("[LIGHTGBM] Splitting contiguous Group-Aware Train/Val datasets...")
    
    val_users = train_matrix_df.select("customer_id").unique().sample(fraction=0.2, seed=42)
    
    val_matrix_df = train_matrix_df.join(val_users, on="customer_id", how="inner")
    train_matrix_df_clean = train_matrix_df.join(val_users, on="customer_id", how="anti")
    
    del train_matrix_df, val_users, label_tx_snapshot; gc.collect()
    
    X_train = train_matrix_df_clean.select(LGBM_FEATURES).to_numpy()
    y_train = train_matrix_df_clean["label"].to_numpy().astype(np.int32)
    group_train = train_matrix_df_clean.group_by("customer_id", maintain_order=True).len()["len"].to_numpy()
    train_dataset = lgb.Dataset(X_train, label=y_train, group=group_train)
    
    X_val = val_matrix_df.select(LGBM_FEATURES).to_numpy()
    y_val = val_matrix_df["label"].to_numpy().astype(np.int32)
    group_val = val_matrix_df.group_by("customer_id", maintain_order=True).len()["len"].to_numpy()
    val_dataset = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_dataset)
    
    del train_matrix_df_clean, val_matrix_df; gc.collect()
    
    lgbm_params = {
        "objective": "lambdarank",
        "metric": "ndcg",
        "ndcg_eval_at": [10],
        "boosting_type": "gbdt",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": -1,  
        "device_type": "gpu", 
        "deterministic": True,
        "random_state": 42,
        "verbose": -1
    }
    print_status("[LIGHTGBM] Training LambdaRank model over GPU acceleration with Early Stopping...")
    ranker = lgb.train(
        lgbm_params, 
        train_dataset, 
        num_boost_round=1000,
        valid_sets=[train_dataset, val_dataset],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.log_evaluation(period=10),
            lgb.early_stopping(stopping_rounds=50)
        ]
    )
    
    # ĐÃ BỔ SUNG: Trích xuất và hiển thị danh sách 10 đặc trưng đóng góp mạnh nhất vào hàm Gain phân nhánh
    importance = ranker.feature_importance(importance_type='gain')
    feature_imp_df = pl.DataFrame({
        "feature": LGBM_FEATURES,
        "importance": importance
    }).sort("importance", descending=True).head(20)
    
    print("\n" + "="*65)
    print(" »»» TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (LIGHTGBM TOTAL GAIN) «««")
    print("="*65)
    for idx, row in enumerate(feature_imp_df.iter_rows(named=True)):
        print(f"  Hạng {idx+1:02d} | Đặc trưng: {row['feature']:<30} | Điểm Gain: {row['importance']:,.4f}")
    print("="*65 + "\n")
    
    del X_train, y_train, group_train, train_dataset; 
    del X_val, y_val, group_val, val_dataset; gc.collect()
    
    # --------------------------------------------------------------------------
    # STAGE 2 - PHASE 3: HUẤN LUYỆN LẠI TRÊN FULL SCALE THÁNG 1-12 ĐỂ DỰ ĐOÁN THÁNG 1 NĂM SAU
    # --------------------------------------------------------------------------
    print_status("[PIPELINE STAGE 2] Overwriting configurations to Full Months 1-12 for January inference...")
    for f in CACHE_DIR.glob("*.parquet"): f.unlink()
    
    channel_files = {} 
    
    hist_tx, target_users = load_history_data()
    if USER_LIMIT > 0:
        target_users = target_users.head(USER_LIMIT)
        hist_tx = hist_tx.filter(pl.col("customer_id").is_in(target_users["customer_id"].to_list()))
    hist_events = load_event_history(target_users)
    
    p_uf, p_if, k_svd = precompute_advanced_features(hist_tx, items)
    profile_paths = compute_user_archetypes_and_profiles(hist_tx, items, target_users)
    profile_paths["user_features"] = p_uf
    profile_paths["item_features"] = p_if
    
    ref_maps = {
        "B_local": channel_local_popular_map(hist_tx, top_k=500),
        "E_cat": channel_category_popular_map(hist_tx, items, bestsellers_per_category=80),
        "F_brand": channel_brand_popular_map(hist_tx, items, bestsellers_per_brand=50),
        "G_global": channel_global_popular_map(hist_tx, global_k=300),
        "H_trend": channel_category_trending_map(hist_tx, items, trending=50),
        "S4_month_pop": channel_s4_monthly_pop_map(hist_tx, n_pop=100),
        "S5_month_trend": channel_s5_monthly_trend_map(hist_tx, n_trend=20)
    }
    
    latent_models = {"cf_latent": train_cf_latent(hist_tx), "als": train_s2_als(hist_tx), "k_svd": k_svd}
    
    df_a = channel_history(hist_tx)
    p_a = CACHE_DIR / "ch_A_history.parquet"; df_a.write_parquet(p_a); channel_files["A_history"] = p_a
    df_s1 = channel_s1_history_recent(hist_tx, hist_events, n_history=200)
    p_s1 = CACHE_DIR / "ch_S1_hist_recent.parquet"; df_s1.write_parquet(p_s1); channel_files["S1_hist_recent"] = p_s1
    df_s6 = channel_s6_full_history(hist_tx, n_full_history=300)
    p_s6 = CACHE_DIR / "ch_S6_full_hist.parquet"; df_s6.write_parquet(p_s6); channel_files["S6_full_hist"] = p_s6
    del df_a, df_s1, df_s6; gc.collect()
    
    del hist_tx, target_users, hist_events; gc.collect()
    print_status("Main transactional graph dropped. Invoking In-Loop LightGBM Chunk Inference Engine...")
    
    final_candidates = evaluate_and_fuse(channel_files, ref_maps, profile_paths, latent_models, max_final_candidates=300, label_tx=None, model=ranker)
    
    print_status("Packaging and dumping final predictions dictionary to pickle...")
    submission_df = final_candidates.group_by("customer_id", maintain_order=True).agg(pl.col("item_id").alias("recommended_items"))
    del final_candidates; gc.collect()
    
    submission_dict = {}
    for row in submission_df.iter_rows():
        submission_dict[row[0]] = row[1]
    del submission_df; gc.collect()
    
    output_filename = "candidates_pir_integrated_v2.pkl"
    with open(output_filename, "wb") as f:
        pickle.dump(submission_dict, f)
        
    print_status("Cleaning temporary disk cache files...")
    all_temp_paths = list(channel_files.values()) + list(profile_paths.values()) + [CACHE_DIR / "lgbm_train_matrix.parquet", CACHE_DIR / "final_recommendations_nhap.parquet"]
    for f_path in all_temp_paths:
        if f_path.exists():
            f_path.unlink()
    CACHE_DIR.rmdir()
    print_status(f"✓ Two-Stage LightGBM Reranking Pipeline successfully finished! Saved clean prediction file at '{output_filename}'")

if __name__ == "__main__":
    run_pipeline()

[15:13:39] [RAM: 4.90 GB] --- Starting Full Feature-Integrated PIR Pipeline v2 (Two-Stage ML Scale) ---
[15:13:39] [RAM: 4.90 GB] Loading items metadata and executing description age parsing text mining...
